# Dogs vs Cats
This notebook runs Part 1: baseline Dogs vs Cats classifiers under identity and fixed tile-wise permutations.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [1]:
import sys, os
from pathlib import Path
import subprocess

make sure installations are made before other imports

In [2]:
def get_project_root():
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parents[1]
    elif project_root.name == 'src':
        project_root = project_root.parent
    print(f"Project root: {project_root}")
    return project_root

In [3]:
REPO_PATH = get_project_root()

Project root: /Users/royrubin/Documents/GitHub/MLDS_Final_Project


### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


before local iports make sure repo is updated

In [4]:
def update_git_repo():
    try:
        # הרצת הפקודה ובדיקה אם היא הצליחה (check=True)
        result = subprocess.run(['git', 'pull'], check=True, capture_output=True, text=True)
        print("Update successful:", result.stdout)
    except subprocess.CalledProcessError as e:
        # אם הפקודה נכשלה, המערכת תדפיס שגיאה ותעצור את ריצת התאים הבאים
        print("Git pull failed!")
        print("Error details:", e.stderr)
        raise Exception("Stopping execution due to Git Pull failure")

In [ ]:
update_git_repo()


Update successful: Already up to date.



: 

make local imports

In [ ]:
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

from src.evaluation.experiment_results import (
    experiment_output_paths,
    get_device,
    load_experiment_samples,
    plot_accuracy_vs_tiles,
    save_aggregated_accuracy,
    save_rows,
)
from src.preprocessing.dogs_cats import build_dataloaders, class_counts
from src.preprocessing.permutations import build_permutation_records
from src.training.experiment_steps import train_model_configuration
from src.utils.config import CVExperimentConfig
from src.utils.io import ensure_dir, save_csv
from src.utils.reproducibility import seed_everything

### Setup configs

In [ ]:
# Config class already imported above

In [ ]:
configs = CVExperimentConfig()
configs


### Global Definitions
Define paths and load the grouped YAML config.


### setup paths in relation to usa

In [ ]:
# Make relevant instalation only if using collab (otherwise, already installed)
os.chdir(REPO_PATH)
%pip install -r requirements.txt

### final imports (after doing pip install if working on colab)

In [ ]:
import json
import random
from IPython.display import Image, display
import pandas as pd
import numpy as np
import torch
import torchvision

## Experiments

In [ ]:
dont forget to sample if config says so

In [ ]:
def part1_output_paths(config):
    """Build stable Part 1 output paths for notebook display."""
    paths = experiment_output_paths(config.results_dir, config.figures_dir, 'part1')
    paths['accuracy_plot'] = paths['figure']
    return paths


def load_part1_data(config, seed=None):
    """Load the Dogs vs Cats train/validation/test split for Part 1."""
    selected_seed = config.seed if seed is None else seed
    samples = load_experiment_samples(config, seed=selected_seed)
    return samples


def build_part1_result_row(config, run_id, model_name, record, seed, metrics):
    """Create one raw Part 1 result row."""
    row = {
        'part': 'part1',
        'run_id': run_id,
        'config_name': config.config_name,
        'model_name': model_name,
        'grid_size': record.grid_size,
        'num_tiles': record.grid_size * record.grid_size,
        'permutation_id': record.permutation_id,
        'permutation_seed': record.permutation_seed,
        'seed': seed,
        **metrics,
    }
    return row


def run_part1_notebook(config):
    """Run Part 1 from this notebook and save raw, aggregated, and figure outputs."""
    ensure_dir(config.results_dir)
    ensure_dir(config.figures_dir)
    device = get_device(config)
    output_paths = part1_output_paths(config)
    run_id = config.config_name
    rows = []

    permutation_records = build_permutation_records(
        grid_sizes=config.grid_sizes,
        num_permutations=config.num_permutations,
        permutation_seed=config.permutation_seed,
        include_identity=True,
    )
    permutation_rows = [record.__dict__ | {'permutation': json.dumps(record.permutation)} for record in permutation_records]
    save_csv(permutation_rows, output_paths['permutations'])

    seed = config.seed
    seed_everything(seed, deterministic=config.deterministic)
    train_samples, validation_samples, _ = load_part1_data(config, seed=seed)
    for model_name in config.model_names:
        for record in permutation_records:
            if record.grid_size == 1 and record.permutation_id > 0:
                continue
            train_loader, validation_loader = build_dataloaders(
                train_samples,
                validation_samples,
                image_size=config.image_size,
                grid_size=record.grid_size,
                permutation=record.permutation,
                seed=seed,
                batch_size=config.batch_size,
                num_workers=config.num_workers,
                standard_augmentation=False,
            )
            metrics = train_model_configuration(
                config,
                model_name,
                train_loader,
                validation_loader,
                device,
                overrides={'pretrained': config.pretrained and model_name != 'convmixer'},
            )
            row = build_part1_result_row(config, run_id, model_name, record, seed, metrics)
            rows.append(row)
            save_rows(rows, output_paths['raw_results'])

    raw_results = pd.DataFrame(rows)
    aggregated_results = save_aggregated_accuracy(
        raw_results,
        ['model_name', 'grid_size', 'num_tiles'],
        output_paths['aggregated_results'],
    )
    plot_accuracy_vs_tiles(aggregated_results, output_paths['accuracy_plot'])
    return aggregated_results

### Setup Exp

In [ ]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [ ]:
output_paths = part1_output_paths(configs)
output_paths

### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
if not configs.using_google_colab:
    configs.sample_data = True
    configs.sample_limit = 1000

train_samples, validation_samples, test_samples = load_part1_data(configs)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print(f'Test samples: {len(test_samples)}')
print('Train class counts:', class_counts(train_samples))

if configs.plot_samples:
    import matplotlib.pyplot as plt

    cat_samples = [sample for sample in train_samples if sample[1] == 0][:3]
    dog_samples = [sample for sample in train_samples if sample[1] == 1][:3]
    sample_pairs = cat_samples + dog_samples

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, (path, label) in enumerate(sample_pairs):
        img = plt.imread(path)
        axes[i].imshow(img)
        axes[i].set_title('Cat' if label == 0 else 'Dog')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

### Experiments - Run Baselines
Run the configured baseline grid/model/permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [ ]:
# Build and save permutation records
permutation_records = build_permutation_records(
    grid_sizes=configs.grid_sizes,
    num_permutations=configs.num_permutations,
    permutation_seed=configs.permutation_seed,
    include_identity=True,
)
permutation_rows = [record.__dict__ | {'permutation': json.dumps(record.permutation)} for record in permutation_records]
save_csv(permutation_rows, output_paths['permutations'])
print(f"Saved {len(permutation_records)} permutation records")

In [ ]:
# Setup reproducibility and load data
seed = configs.seed
seed_everything(seed, deterministic=configs.deterministic)
train_samples, validation_samples, _ = load_part1_data(configs, seed=seed)
print(f"Loaded {len(train_samples)} train, {len(validation_samples)} val samples")

In [ ]:
### Train ResNet18

In [ ]:
# Prepare shared result accumulation across model runs
all_rows = []

In [ ]:
model_name = "resnet18"
device = get_device(configs)
rows = []
run_id = configs.config_name

for record in permutation_records:
    if record.grid_size == 1 and record.permutation_id > 0:
        continue
    train_loader, validation_loader = build_dataloaders(
        train_samples,
        validation_samples,
        image_size=configs.image_size,
        grid_size=record.grid_size,
        permutation=record.permutation,
        seed=seed,
        batch_size=configs.batch_size,
        num_workers=configs.num_workers,
        standard_augmentation=False,
    )
    metrics = train_model_configuration(
        configs,
        model_name,
        train_loader,
        validation_loader,
        device,
        overrides={'pretrained': configs.pretrained and model_name != 'convmixer'},
    )
    row = build_part1_result_row(configs, run_id, model_name, record, seed, metrics)
    rows.append(row)

all_rows.extend(rows)
save_rows(all_rows, output_paths['raw_results'])
print(f"Completed training ResNet18 with {len(rows)} runs")

In [ ]:
model_name = "swin_t"
rows = []

for record in permutation_records:
    if record.grid_size == 1 and record.permutation_id > 0:
        continue
    train_loader, validation_loader = build_dataloaders(
        train_samples,
        validation_samples,
        image_size=configs.image_size,
        grid_size=record.grid_size,
        permutation=record.permutation,
        seed=seed,
        batch_size=configs.batch_size,
        num_workers=configs.num_workers,
        standard_augmentation=False,
    )
    metrics = train_model_configuration(
        configs,
        model_name,
        train_loader,
        validation_loader,
        device,
        overrides={'pretrained': configs.pretrained and model_name != 'convmixer'},
    )
    row = build_part1_result_row(configs, run_id, model_name, record, seed, metrics)
    rows.append(row)

all_rows.extend(rows)
save_rows(all_rows, output_paths['raw_results'])
print(f"Completed training Swin-T with {len(rows)} runs")

In [ ]:
model_name = "convmixer"
rows = []

for record in permutation_records:
    if record.grid_size == 1 and record.permutation_id > 0:
        continue
    train_loader, validation_loader = build_dataloaders(
        train_samples,
        validation_samples,
        image_size=configs.image_size,
        grid_size=record.grid_size,
        permutation=record.permutation,
        seed=seed,
        batch_size=configs.batch_size,
        num_workers=configs.num_workers,
        standard_augmentation=False,
    )
    metrics = train_model_configuration(
        configs,
        model_name,
        train_loader,
        validation_loader,
        device,
        overrides={'pretrained': configs.pretrained and model_name != 'convmixer'},
    )
    row = build_part1_result_row(configs, run_id, model_name, record, seed, metrics)
    rows.append(row)

all_rows.extend(rows)
save_rows(all_rows, output_paths['raw_results'])
print(f"Completed training ConvMixer with {len(rows)} runs")

In [ ]:
# Aggregate results and plot
raw_results = pd.read_csv(output_paths['raw_results'])
aggregated_results = save_aggregated_accuracy(
    raw_results,
    ['model_name', 'grid_size', 'num_tiles'],
    output_paths['aggregated_results'],
)
plot_accuracy_vs_tiles(aggregated_results, output_paths['accuracy_plot'])
aggregated_results

### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk.


In [ ]:
saved_results = {
    'raw': pd.read_csv(output_paths['raw_results']),
    'aggregated': pd.read_csv(output_paths['aggregated_results']),
}
display(saved_results['aggregated'])

### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
figure_path = output_paths['accuracy_plot']
if Path(figure_path).exists():
    display(Image(filename=figure_path))
else:
    print(f'Plot not found yet: {figure_path}')
